In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import mlflow
import time
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

pd.set_option('display.max_columns', 200)

mlflow.set_tracking_uri("file:../mlruns")
mlflow.set_experiment("loan-default")

print(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")
print(f"Experiment: {mlflow.get_experiment_by_name('loan-default').name}")

In [ ]:
df = pd.read_parquet('../data/loans_clean.parquet')

# Apply EDA cleaning
df['dti'] = df['dti'].replace([-1, 999], np.nan).clip(upper=100)
df['credit_history_months'] = df['credit_history_months'].replace(999, np.nan)
df['annual_inc'] = df['annual_inc'].clip(upper=1_000_000)
df['revol_util'] = df['revol_util'].clip(upper=100)

# Temporal split
df_train = df[df['issue_year'].isin([2014, 2015])].copy()
df_val   = df[df['issue_year'] == 2016].copy()
df_test  = df[df['issue_year'] == 2017].copy()

y_train = df_train['default'].values
y_val   = df_val['default'].values

print(f"Train: {len(df_train):,}  |  Val: {len(df_val):,}  |  Test: {len(df_test):,}")

# Cost matrix function
GAIN_RATE = 0.10
LOSS_RATE = 0.50

def calculate_profit(df_subset, approval_mask):
    approved = df_subset[approval_mask]
    if len(approved) == 0:
        return 0
    profit = np.where(
        approved['default'] == 0,
         GAIN_RATE * approved['loan_amnt'],
        -LOSS_RATE * approved['loan_amnt']
    )
    return profit.sum()

In [ ]:
def prep_features(df):
    """Apply EDA-driven feature engineering decisions."""
    df = df.copy()
    redundant = ['fico_range_high', 'funded_amnt', 'funded_amnt_inv',
                 'num_sats', 'installment', 'num_rev_tl_bal_gt_0']
    joint_cols = [c for c in df.columns if c.startswith('sec_app_') or c.endswith('_joint')]
    high_cardinality = ['zip_code', 'sub_grade']
    split_cols = ['issue_year']
    cols_to_drop = redundant + joint_cols + high_cardinality + split_cols
    df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])
    
    emp_map = {f'{i} year{"s" if i != 1 else ""}': i for i in range(1, 10)}
    emp_map['< 1 year'] = 0
    emp_map['10+ years'] = 10
    df['emp_length'] = df['emp_length'].map(emp_map)
    return df


def build_preprocessor(numeric_cols, categorical_cols):
    """Build the same preprocessor we'll use for every model."""
    numeric_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])
    categorical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
        ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])
    return ColumnTransformer(
        transformers=[
            ('num', numeric_transformer, numeric_cols),
            ('cat', categorical_transformer, categorical_cols)
        ]
    )


# Apply prep, separate target, parse term
df_train_fe = prep_features(df_train)
df_val_fe   = prep_features(df_val)

X_train = df_train_fe.drop(columns=['default'])
X_val   = df_val_fe.drop(columns=['default'])

X_train['term'] = X_train['term'].str.extract(r'(\d+)').astype(int)
X_val['term']   = X_val['term'].str.extract(r'(\d+)').astype(int)

categorical_cols = X_train.select_dtypes(include=['object']).columns.tolist()
numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()

print(f"Train shape: {X_train.shape}")
print(f"Categorical: {len(categorical_cols)} cols, Numeric: {len(numeric_cols)} cols")

In [ ]:
# Build the preprocessor
preprocessor_rf = build_preprocessor(numeric_cols, categorical_cols)

rf_baseline = Pipeline(steps=[
    ('preprocessor', preprocessor_rf),
    ('classifier', RandomForestClassifier(
        n_estimators=200,
        max_depth=20,
        min_samples_leaf=20,
        class_weight='balanced',
        n_jobs=-1,
        random_state=42
    ))
])

with mlflow.start_run(run_name="rf_baseline"):
    # Log parameters
    mlflow.log_param("model_type", "RandomForestClassifier")
    mlflow.log_param("n_estimators", 200)
    mlflow.log_param("max_depth", 20)
    mlflow.log_param("min_samples_leaf", 20)
    mlflow.log_param("class_weight", "balanced")
    mlflow.log_param("n_features_input", X_train.shape[1])
    
    # Train
    print("Training Random Forest (expect 2-5 min)...")
    start = time.time()
    rf_baseline.fit(X_train, y_train)
    train_time = time.time() - start
    print(f"Done in {train_time:.1f}s")
    
    # Predict
    val_probs = rf_baseline.predict_proba(X_val)[:, 1]
    
    # Core metrics
    auc = roc_auc_score(y_val, val_probs)
    pr_auc = average_precision_score(y_val, val_probs)
    brier = brier_score_loss(y_val, val_probs)
    
    # Profit optimization
    thresholds = np.linspace(0.05, 0.55, 100)
    profits = [calculate_profit(df_val, val_probs < t) for t in thresholds]
    best_idx = int(np.argmax(profits))
    best_threshold = float(thresholds[best_idx])
    best_profit = float(profits[best_idx])
    approval_rate = float((val_probs < best_threshold).mean())
    
    # Log
    mlflow.log_param("best_threshold", best_threshold)
    mlflow.log_metric("auc", auc)
    mlflow.log_metric("pr_auc", pr_auc)
    mlflow.log_metric("brier", brier)
    mlflow.log_metric("profit_at_threshold", best_profit)
    mlflow.log_metric("approval_rate", approval_rate)
    mlflow.log_metric("train_time_seconds", train_time)
    
    mlflow.sklearn.log_model(rf_baseline, "model")
    
    print(f"\n=== RANDOM FOREST (baseline) ===")
    print(f"Train time:     {train_time:.1f}s")
    print(f"AUC:            {auc:.4f}")
    print(f"PR-AUC:         {pr_auc:.4f}")
    print(f"Brier:          {brier:.4f}")
    print(f"Best threshold: {best_threshold:.4f}")
    print(f"Approval rate:  {approval_rate:.2%}")
    print(f"Profit:         ${best_profit:,.0f}")
    
    print(f"\n--- Comparison to LogReg ---")
    print(f"LogReg:    AUC 0.7157, Profit $60.7M")
    print(f"RF:        AUC {auc:.4f}, Profit ${best_profit/1e6:.1f}M")
    print(f"Delta:     AUC {auc - 0.7157:+.4f}, Profit ${(best_profit - 60_676_972)/1e6:+.1f}M")

In [ ]:
# RF without class balancing
rf_no_balance = Pipeline(steps=[
    ('preprocessor', build_preprocessor(numeric_cols, categorical_cols)),
    ('classifier', RandomForestClassifier(
        n_estimators=200,
        max_depth=20,
        min_samples_leaf=20,
        class_weight=None,   # <-- the change
        n_jobs=-1,
        random_state=42
    ))
])

with mlflow.start_run(run_name="rf_no_balance"):
    mlflow.log_param("model_type", "RandomForestClassifier")
    mlflow.log_param("n_estimators", 200)
    mlflow.log_param("max_depth", 20)
    mlflow.log_param("min_samples_leaf", 20)
    mlflow.log_param("class_weight", "None")
    mlflow.log_param("n_features_input", X_train.shape[1])
    
    print("Training RF (no class balancing)...")
    start = time.time()
    rf_no_balance.fit(X_train, y_train)
    train_time = time.time() - start
    
    val_probs = rf_no_balance.predict_proba(X_val)[:, 1]
    
    auc = roc_auc_score(y_val, val_probs)
    pr_auc = average_precision_score(y_val, val_probs)
    brier = brier_score_loss(y_val, val_probs)
    
    thresholds = np.linspace(0.05, 0.55, 100)
    profits = [calculate_profit(df_val, val_probs < t) for t in thresholds]
    best_idx = int(np.argmax(profits))
    best_threshold = float(thresholds[best_idx])
    best_profit = float(profits[best_idx])
    approval_rate = float((val_probs < best_threshold).mean())
    
    mlflow.log_param("best_threshold", best_threshold)
    mlflow.log_metric("auc", auc)
    mlflow.log_metric("pr_auc", pr_auc)
    mlflow.log_metric("brier", brier)
    mlflow.log_metric("profit_at_threshold", best_profit)
    mlflow.log_metric("approval_rate", approval_rate)
    mlflow.log_metric("train_time_seconds", train_time)
    
    mlflow.sklearn.log_model(rf_no_balance, "model")
    
    print(f"\n=== RF (no class_weight) ===")
    print(f"Train time:     {train_time:.1f}s")
    print(f"AUC:            {auc:.4f}")
    print(f"PR-AUC:         {pr_auc:.4f}")
    print(f"Brier:          {brier:.4f}")
    print(f"Best threshold: {best_threshold:.4f}")
    print(f"Approval rate:  {approval_rate:.2%}")
    print(f"Profit:         ${best_profit:,.0f}")
    
    print(f"\n--- Comparison ---")
    print(f"RF balanced:   AUC 0.7158, Brier 0.1910, Profit $60.4M")
    print(f"RF unbalanced: AUC {auc:.4f}, Brier {brier:.4f}, Profit ${best_profit/1e6:.1f}M")